# RFW Phase 2 — Full-Scale VLM Hallucination Study

**Models**: Qwen2.5-VL-7B-Instruct · LLaVA-NeXT-Mistral-7B  
**Sample**: 4 ethnicities × 60 pairs (30 genuine + 30 impostor) = 240 pairs per model  
**Prompt variants**: 2 (direct feature-comparison, chain-of-thought)  
**Total generations**: 240 × 2 × 2 = **960**

The pair list is sampled once in §5 and written to `pairs_locked.csv`. Both model blocks
read from that file — neither ever resamples. This makes the cross-model comparison in
Phase 5 depend on a concrete, inspectable artifact, not on two independent seed-42 calls
producing the same result across different execution environments.

Each model block (§7–§8 and §9–§10) can be run as its own Kaggle session if runtime risks
hitting the session cap (~9 h). Both sessions need `pairs_locked.csv` available; upload it
to a Kaggle dataset or attach it as a session input before starting the second session.

| Output | Path |
|--------|------|
| Locked pair list | `/kaggle/working/pairs_locked.csv` |
| Qwen results | `/kaggle/working/qwen_full_results.csv` |
| LLaVA-NeXT results | `/kaggle/working/llava_next_full_results.csv` |

## 1 — Install Dependencies

Torch and CUDA are pre-installed on the Kaggle base image.

In [2]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "transformers>=4.49.0",
    "accelerate>=0.34.0",
    "qwen-vl-utils",
    "pillow",
    "pandas",
)
print("Packages ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 56.2 MB/s eta 0:00:00
Packages ready.


## 2 — Imports & Global Config

Edit `DATASET_ROOT` before running.

In [3]:
import gc
import re
import random
import time
import datetime
from contextlib import contextmanager
from pathlib import Path

import pandas as pd
import torch
from PIL import Image

DATASET_ROOT = Path("/kaggle/input/datasets/chidroopakanaparthy/rfw-clean-400x400")

IMAGE_ROOT = DATASET_ROOT / "data"
TXT_ROOT   = DATASET_ROOT / "txts"

PAIRS_LOCKED_CSV = Path("/kaggle/working/pairs_locked.csv")
QWEN_OUTPUT_CSV  = Path("/kaggle/working/qwen_full_results.csv")
LLAVA_OUTPUT_CSV = Path("/kaggle/working/llava_next_full_results.csv")

ETHNICITIES        = ["African", "Asian", "Caucasian", "Indian"]
N_GENUINE_PER_ETH  = 30
N_IMPOSTOR_PER_ETH = 30
RANDOM_SEED        = 42
MAX_NEW_TOKENS     = 500
# Save checkpoint every SAVE_EVERY new rows appended.
# /kaggle/working/ persists across in-session kernel restarts but NOT across
# a full new Kaggle session (new container). SAVE_EVERY=10 limits loss to
# at most 10 rows if a session dies; a session restart within the same
# kernel loses nothing because the CSV is already on disk.
SAVE_EVERY     = 10
PROGRESS_EVERY = 20

DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)

print("Device:", "GPU — " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("dtype :", DTYPE)

Device: GPU — Tesla T4
dtype : torch.float16


## 3 — Mount Diagnostic

In [4]:
def print_tree(root: Path, max_depth=3, max_items=8, _depth=0):
    indent = "  " * _depth
    if not root.exists():
        print(f"{indent}[NOT FOUND] {root}")
        return
    children = sorted(root.iterdir())
    for child in children[:max_items]:
        tag = "/" if child.is_dir() else ""
        print(f"{indent}{child.name}{tag}")
        if child.is_dir() and _depth + 1 < max_depth:
            print_tree(child, max_depth=max_depth, max_items=4, _depth=_depth + 1)
    if len(children) > max_items:
        print(f"{indent}... ({len(children) - max_items} more)")

print(f"Dataset tree (root: {DATASET_ROOT}):")
print_tree(DATASET_ROOT)
print()

for eth in ETHNICITIES:
    img_dir    = IMAGE_ROOT / eth
    pairs_file = TXT_ROOT  / eth / f"{eth}_pairs.txt"
    print(f"  {eth:10s}  images={'OK' if img_dir.is_dir() else 'MISSING'}  "
          f"pairs={'OK' if pairs_file.exists() else 'MISSING'}")

Dataset tree (root: /kaggle/input/datasets/chidroopakanaparthy/rfw-clean-400x400):
CHANGELOG.txt
data/
  African/
    m.010lz5/
    m.011y5k/
    m.01257s/
    m.0126y2/
    ... (2991 more)
  Asian/
    m.0112yl/
    m.01167d3z/
    m.012fgx/
    m.012t8n/
    ... (2488 more)
  Caucasian/
    m.011_2h/
    m.011xl3/
    m.012034/
    m.0125mr/
    ... (2954 more)
  Indian/
    m.010g87/
    m.01241v/
    m.012g8l/
    m.0138r2/
    ... (2980 more)
txts/
  African/
    African_images.txt
    African_lmk.txt
    African_pairs.txt
    African_people.txt
  Asian/
    Asian_images.txt
    Asian_lmk.txt
    Asian_pairs.txt
    Asian_people.txt
  Caucasian/
    Caucasian_images.txt
    Caucasian_lmk.txt
    Caucasian_pairs.txt
    Caucasian_people.txt
  Indian/
    Indian_images.txt
    Indian_lmk.txt
    Indian_pairs.txt
    Indian_people.txt

  African     images=OK  pairs=OK
  Asian       images=OK  pairs=OK
  Caucasian   images=OK  pairs=OK
  Indian      images=OK  pairs=OK


## 4 — Prompt Variants

Exact texts validated in the pilot.

In [5]:
PROMPT_VARIANT_1 = (
    "Look at these two face images. Describe the specific facial "
    "features that support or contradict them being the same person, "
    "then state your final answer as Yes or No."
)

PROMPT_VARIANT_2 = (
    "Look carefully at Image 1 and Image 2. "
    "Step 1: Describe the key facial features you observe in each image "
    "(e.g. face shape, eye region, nose, mouth, skin tone, hair). "
    "Step 2: Compare the features across both images and note similarities "
    "and differences. "
    "Step 3: Based on your analysis, decide whether the two faces belong "
    "to the same person and state your final answer as Yes or No."
)

PROMPT_VARIANTS = {
    1: ("direct_feature_comparison", PROMPT_VARIANT_1),
    2: ("chain_of_thought",           PROMPT_VARIANT_2),
}

for vnum, (vlabel, vtext) in PROMPT_VARIANTS.items():
    print(f"Variant {vnum} [{vlabel}]: {vtext[:100]}...")

Variant 1 [direct_feature_comparison]: Look at these two face images. Describe the specific facial features that support or contradict them...
Variant 2 [chain_of_thought]: Look carefully at Image 1 and Image 2. Step 1: Describe the key facial features you observe in each ...


## 5 — Pair Sampling (runs once — output is `pairs_locked.csv`)

The pair list is generated here and immediately written to disk. Both model blocks load this file rather than resampling, making the cross-model comparison immune to environment differences that could cause `random.Random(42)` to produce different sequences in separate sessions.

If running §7–§8 and §9–§10 as separate Kaggle sessions, upload `pairs_locked.csv` as a dataset input to the second session — do **not** rerun this cell.

In [6]:
# ---------------------------------------------------------------------------
# Dual verifier scoring: ArcFace (buffalo_m) and AdaFace (R50/WebFace4M)
#
# Produces two margin columns in pairs_locked.csv:
#   verifier_score_margin_arcface  = cosine_sim(e1, e2) - arcface_threshold
#   verifier_score_margin_adaface  = cosine_sim(e1, e2) - adaface_threshold
# Positive margin = verifier predicts same person; negative = different.
#
# THRESHOLDS:
# Both thresholds are sourced from the project's own baseline calibration
# (rfw_fairness_baseline.ipynb, Cell 6 — "COMPUTE METRICS PER RACE PER MODEL" /
# summary_metrics.csv).
# They are fixed, race-agnostic thresholds derived by pooling cosine similarity
# scores across all four races for each model, computing the ROC curve, and
# selecting the highest threshold satisfying FPR <= 1% on the pooled data
# (walk-back-to-last-valid-index method). This single threshold is applied
# uniformly across all races so that any observed disparity reflects model
# behavior, not a threshold artifact.
# ---------------------------------------------------------------------------
import subprocess, sys

def _ensure(*pkgs):
    for pkg in pkgs:
        mod = pkg.split("[")[0].replace("-", "_")
        try:
            __import__(mod)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("insightface", "onnxruntime")

from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from insightface.app import FaceAnalysis

# ---------------------------------------------------------------------------
# Project-calibrated thresholds from rfw_fairness_baseline.ipynb
# ---------------------------------------------------------------------------
ARCFACE_THRESHOLD = 0.3133
ADAFACE_THRESHOLD = 0.3259

# ---------------------------------------------------------------------------
# ArcFace: InsightFace buffalo_m (SCRFD-2.5GF detector + ResNet50@WebFace600K)
# ---------------------------------------------------------------------------
print("Loading insightface buffalo_m ...")
import os
import shutil
from insightface.utils.storage import download
# Fix for insightface nested extraction bug (buffalo_m/buffalo_m)
download("models", "buffalo_m", root="~/.insightface")
model_dir = os.path.expanduser("~/.insightface/models/buffalo_m")
nested_dir = os.path.join(model_dir, "buffalo_m")
if os.path.exists(nested_dir):
    for f in os.listdir(nested_dir):
        shutil.move(os.path.join(nested_dir, f), model_dir)
    shutil.rmtree(nested_dir)

arcface_app = FaceAnalysis(
    name="buffalo_m",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
arcface_app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.1)
print("  buffalo_m ready.")


def get_arcface_embedding(img_path: str):
    img = cv2.imread(img_path)
    if img is None:
        return None
    faces = arcface_app.get(img)
    if not faces:
        return None
    face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    return face.normed_embedding   # L2-normalised; dot product = cosine similarity


# ---------------------------------------------------------------------------
# AdaFace: R50/WebFace4M
#
# Checkpoint: download adaface_ir50_webface4m.ckpt from the official Google
# Drive link in mk-minchul/AdaFace README and place it at:
#   /kaggle/input/adaface-weights/adaface_ir50_webface4m.ckpt
# Input format: BGR, 112x112, normalised to [-1, 1] ((x/255 - 0.5) / 0.5).
# Embeddings are L2-normalised after extraction so dot product = cosine sim.
# Architecture loaded via the net module from mk-minchul/AdaFace (install below).
# ---------------------------------------------------------------------------

ADAFACE_CKPT = Path("/kaggle/input/datasets/chidroopakanaparthy/adaface-weights/adaface_ir50_webface4m.ckpt")

if not ADAFACE_CKPT.exists():
    raise FileNotFoundError(
        f"AdaFace checkpoint not found: {ADAFACE_CKPT}\n"
        "Download adaface_ir50_webface4m.ckpt from the Google Drive link in the "
        "mk-minchul/AdaFace README and attach it as a Kaggle dataset input."
    )

# Load via official net module pattern (confirmed from mk-minchul/AdaFace/inference.py)
try:
    import net as adaface_net
except ImportError:
    # Fallback: clone the repo on the fly (Kaggle has internet enabled)
    subprocess.check_call(["git", "clone", "--depth=1",
                           "https://github.com/mk-minchul/AdaFace.git",
                           "/kaggle/working/AdaFace"])
    sys.path.insert(0, "/kaggle/working/AdaFace")
    import net as adaface_net

print("Loading AdaFace R50/WebFace4M ...")
adaface_model = adaface_net.build_model("ir_50")
state = torch.load(ADAFACE_CKPT, map_location="cpu")["state_dict"]
state = {k[6:]: v for k, v in state.items() if k.startswith("model.")}
adaface_model.load_state_dict(state)
adaface_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
adaface_model = adaface_model.to(adaface_device).eval()
print("  AdaFace R50/WebFace4M ready.")

_ADA_MEAN = torch.tensor([0.5, 0.5, 0.5], device=adaface_device).view(1, 3, 1, 1)
_ADA_STD  = torch.tensor([0.5, 0.5, 0.5], device=adaface_device).view(1, 3, 1, 1)


def get_adaface_embedding(img_path: str):
    # Detect face with insightface (shares detector with ArcFace for consistency).
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None
    faces = arcface_app.get(img_bgr)
    if not faces:
        return None
    face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))

    # Crop and align to 112x112 using the 5-point keypoints from insightface.
    from insightface.utils import face_align
    aligned_bgr = face_align.norm_crop(img_bgr, landmark=face.kps, image_size=112)
    # BGR -> [0,1] -> [-1,1] (AdaFace expects BGR)
    aligned_input = aligned_bgr.astype(np.float32) / 255.0
    t = torch.from_numpy(aligned_input).permute(2, 0, 1).unsqueeze(0).to(adaface_device)
    t = (t - _ADA_MEAN) / _ADA_STD

    with torch.no_grad():
        emb, _ = adaface_model(t)
    emb = F.normalize(emb, p=2, dim=1)
    return emb[0].cpu().numpy()


# ---------------------------------------------------------------------------
# Score all pairs
# ---------------------------------------------------------------------------
print(f"\nScoring {len(sampled_pairs)} pairs with both verifiers ...")
arc_no_face, ada_no_face = 0, 0

for pair in sampled_pairs:
    e1_arc = get_arcface_embedding(pair["image1_path"])
    e2_arc = get_arcface_embedding(pair["image2_path"])
    if e1_arc is None or e2_arc is None:
        pair["verifier_score_margin_arcface"] = None
        arc_no_face += 1
    else:
        sim = float(np.dot(e1_arc, e2_arc))
        pair["verifier_score_margin_arcface"] = round(sim - ARCFACE_THRESHOLD, 6)

    e1_ada = get_adaface_embedding(pair["image1_path"])
    e2_ada = get_adaface_embedding(pair["image2_path"])
    if e1_ada is None or e2_ada is None:
        pair["verifier_score_margin_adaface"] = None
        ada_no_face += 1
    else:
        sim = float(np.dot(e1_ada, e2_ada))
        pair["verifier_score_margin_adaface"] = round(sim - ADAFACE_THRESHOLD, 6)

n_total        = len(sampled_pairs)
n_arc_scored   = sum(1 for p in sampled_pairs if p["verifier_score_margin_arcface"] is not None)
n_ada_scored   = sum(1 for p in sampled_pairs if p["verifier_score_margin_adaface"] is not None)

print(f"ArcFace  scored: {n_arc_scored}/{n_total} ({100*n_arc_scored/n_total:.1f}%)"
      + (f"  [{arc_no_face} face-detection failures]" if arc_no_face else ""))
print(f"AdaFace  scored: {n_ada_scored}/{n_total} ({100*n_ada_scored/n_total:.1f}%)"
      + (f"  [{ada_no_face} face-detection failures]" if ada_no_face else ""))

assert n_arc_scored == n_total, (
    f"ArcFace margin is null for {n_total - n_arc_scored} pair(s). "
    "Inspect face-detection failures before writing pairs_locked.csv."
)
assert n_ada_scored == n_total, (
    f"AdaFace margin is null for {n_total - n_ada_scored} pair(s). "
    "Inspect face-detection failures before writing pairs_locked.csv."
)

# ---------------------------------------------------------------------------
# Write pairs_locked.csv with both margin columns
# ---------------------------------------------------------------------------
import pandas as pd
df_pairs = pd.DataFrame(sampled_pairs)[
    ["pair_id", "ethnicity", "image1_filename", "image2_filename",
     "label", "verifier_score_margin_arcface", "verifier_score_margin_adaface",
     "image1_path", "image2_path"]
].rename(columns={"label": "ground_truth_label"})

df_pairs.to_csv(PAIRS_LOCKED_CSV, index=False)

TOTAL_PAIRS    = len(df_pairs)
TOTAL_VARIANTS = len(PROMPT_VARIANTS)
TOTAL_GENS     = TOTAL_PAIRS * TOTAL_VARIANTS

print(f"\nLocked {TOTAL_PAIRS} pairs to {PAIRS_LOCKED_CSV}")
print(f"Gens per model: {TOTAL_GENS}")

del arcface_app, adaface_model
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Verifier models unloaded.")

ERROR: Operation cancelled by user


KeyboardInterrupt: 

## 6 — Shared Helper Functions

In [7]:
# ---------------------------------------------------------------------------
# Verdict detection — two-tier, applied at generation time.
# Tier 1: literal \b(yes|no)\b.
# Tier 2: semantic paraphrase scoped to the last ~300 chars, where CoT models
# place their final conclusion. Only detector in the notebook; no literal-only
# fallback that would produce a stale status column.
# ---------------------------------------------------------------------------
def has_verdict_v2(text: str) -> dict:
    t = text.strip()

    if re.search(r"\b(yes)\b", t, re.IGNORECASE):
        return {"verdict_detected": True,  "verdict_tier": "literal",  "verdict_polarity": "same"}
    if re.search(r"\b(no)\b",  t, re.IGNORECASE):
        return {"verdict_detected": True,  "verdict_tier": "literal",  "verdict_polarity": "different"}

    tail = t[-300:]
    neg_pats = [
        r"\bnot\s+the\s+same\b",
        r"\btwo\s+different\s+(individuals|people|persons)\b",
        r"\bdifferent\s+(individuals|people|persons)\b",
        r"\bdo(es)?\s+not\s+appear\s+to\s+be\s+the\s+same\b",
    ]
    aff_pats = [
        r"\bsame\s+(person|individual)\b",
        r"\bbelong(s)?\s+to\s+the\s+same\s+person\b",
        r"\bare\s+(likely\s+)?the\s+same\b",
        r"\bappear(s)?\s+to\s+be\s+the\s+same\b",
    ]
    for pat in neg_pats:
        if re.search(pat, tail, re.IGNORECASE):
            return {"verdict_detected": True,  "verdict_tier": "semantic", "verdict_polarity": "different"}
    for pat in aff_pats:
        if re.search(pat, tail, re.IGNORECASE):
            return {"verdict_detected": True,  "verdict_tier": "semantic", "verdict_polarity": "same"}

    return {"verdict_detected": False, "verdict_tier": None, "verdict_polarity": None}


def output_is_truncated(text: str) -> bool:
    return not bool(re.search(r"[.!?]\s*$", text.strip()[-15:]))


def compute_status(verdict_detected, verdict_tier, is_trunc) -> str:
    # Single authority for status — nothing else in this notebook sets it.
    if verdict_detected and verdict_tier == "literal":
        return "completed_literal"
    if verdict_detected and verdict_tier == "semantic":
        return "completed_semantic"
    if is_trunc:
        return "no_verdict_truncated"
    return "no_verdict_ambiguous"


def check_coherence(text: str) -> bool:
    words = text.split()
    if len(words) < 5:
        return False
    if len(set(text[:100].replace(" ", ""))) < 10:
        return False
    if len(words) >= 4:
        bigrams = [(words[i], words[i+1]) for i in range(len(words)-1)]
        if len(set(bigrams)) / len(bigrams) < 0.25:
            return False
    if re.search(r"\b(\w+)(?:\s+\1){14,}\b", text, re.IGNORECASE):
        return False
    return True


def load_completed_keys(csv_path: Path):
    # Loads prior rows so a kernel restart resumes rather than overwrites.
    if not csv_path.exists():
        return [], set()
    df_existing = pd.read_csv(csv_path)
    rows = df_existing.to_dict("records")
    keys = {(int(r["pair_id"]), int(r["prompt_variant"])) for r in rows}
    print(f"  Resume: {len(rows)} existing rows in {csv_path.name}")
    return rows, keys


def load_locked_pairs(csv_path: Path) -> list:
    # Both model blocks call this. The sampling RNG is not touched again.
    df = pd.read_csv(csv_path)
    assert len(df) == 240, f"pairs_locked.csv: expected 240 rows, got {len(df)}"
    expected_ids = list(range(1, 241))
    actual_ids   = sorted(df["pair_id"].tolist())
    assert actual_ids == expected_ids, (
        f"pair_id mismatch: first divergence at index "
        f"{next(i for i,(a,b) in enumerate(zip(actual_ids, expected_ids)) if a != b)}"
    )
    return df.to_dict("records")


print("Shared helpers defined.")

Shared helpers defined.


---
## 7 — QWEN2.5-VL-7B-INSTRUCT: Load, Run, Unload

> **Independent session note**: This block (§7–§8) can be run as its own Kaggle session.
> Prerequisite: `pairs_locked.csv` must be available at `/kaggle/working/pairs_locked.csv`
> or mounted as a dataset input. Do not rerun §5 in a second session — load the locked file
> only. The checkpoint CSV (`qwen_full_results.csv`) lives in `/kaggle/working/`, which
> persists across in-session kernel restarts but is lost on a full new session (new
> container). With `SAVE_EVERY=10` the maximum loss is 10 rows per session failure.

In [18]:
# QWEN_MEAN_SEC is computed from the Qwen pilot results CSV, not guessed.
# Expected path (Kaggle): /kaggle/input/rfw-pilot-outputs/qwen_pilot_results.csv
# Column used: inference_time_seconds, filtered to non-error rows.
# If the file isn't mounted, the cell falls back with a visible warning.

QWEN_PILOT_CSV = Path("/kaggle/input/rfw-pilot-outputs/qwen_pilot_results.csv")
QWEN_FALLBACK_SEC = 20.0   # documented fallback; replace with real value if CSV unavailable

if QWEN_PILOT_CSV.exists():
    _df_qp = pd.read_csv(QWEN_PILOT_CSV)
    # Pilot schema has an "error" column; filter to successful rows only
    if "error" in _df_qp.columns:
        _df_qp = _df_qp[_df_qp["error"] == False]
    QWEN_MEAN_SEC = round(_df_qp["inference_time_seconds"].dropna().mean(), 2)
    print(f"QWEN_MEAN_SEC = {QWEN_MEAN_SEC}s  (computed from {QWEN_PILOT_CSV.name}, "
          f"n={len(_df_qp)} rows)")
else:
    QWEN_MEAN_SEC = QWEN_FALLBACK_SEC
    print(f"WARNING: Pilot CSV not found at {QWEN_PILOT_CSV}")
    print(f"  Using fallback QWEN_MEAN_SEC = {QWEN_MEAN_SEC}s — mount the pilot output "
          f"dataset or update QWEN_PILOT_CSV to get a real estimate.")

pairs_for_estimate  = load_locked_pairs(PAIRS_LOCKED_CSV)
total_gens_estimate = len(pairs_for_estimate) * len(PROMPT_VARIANTS)
est_done            = len(load_completed_keys(QWEN_OUTPUT_CSV)[1])
est_hours           = (total_gens_estimate * QWEN_MEAN_SEC) / 3600
est_remaining_h     = ((total_gens_estimate - est_done) * QWEN_MEAN_SEC) / 3600

print(f"\nQwen: {total_gens_estimate} total generations @ {QWEN_MEAN_SEC:.1f}s each")
print(f"  Full run estimate : {est_hours:.1f} h")
print(f"  Already done      : {est_done}")
print(f"  Remaining estimate: {est_remaining_h:.1f} h")
print(f"  Checkpoint cadence: every {SAVE_EVERY} rows")

  Using fallback QWEN_MEAN_SEC = 20.0s — mount the pilot output dataset or update QWEN_PILOT_CSV to get a real estimate.

Qwen: 480 total generations @ 20.0s each
  Full run estimate : 2.7 h
  Already done      : 0
  Remaining estimate: 2.7 h
  Checkpoint cadence: every 10 rows


In [19]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

print(f"Loading {QWEN_MODEL_ID} ...")
t0 = time.time()

qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
qwen_model.eval()

qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)

print(f"Qwen loaded in {time.time() - t0:.1f}s")

Loading Qwen/Qwen2.5-VL-7B-Instruct ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Qwen loaded in 152.7s


In [22]:
def qwen_generate(model, processor, img1_path, img2_path, prompt_text):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert facial analysis assistant. "
                "Analyse the provided face images carefully and answer the user question."
            ),
        },
        {
            "role": "user",
            "content": [
                {"type": "text",  "text": "Image 1:"},
                {"type": "image", "image": img1_path},
                {"type": "text",  "text": "Image 2:"},
                {"type": "image", "image": img2_path},
                {"type": "text",  "text": prompt_text},
            ],
        },
    ]

    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    t_start = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )
    elapsed = time.perf_counter() - t_start

    new_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]
    out = processor.batch_decode(
        new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

    return out, elapsed

In [23]:
# Reads pairs_locked.csv — does not touch random.Random or resample.
sampled_pairs = load_locked_pairs(PAIRS_LOCKED_CSV)
TOTAL_PAIRS    = len(sampled_pairs)
TOTAL_VARIANTS = len(PROMPT_VARIANTS)
TOTAL_GENS     = TOTAL_PAIRS * TOTAL_VARIANTS

# Resume: load any rows already written in a prior run of this cell.
qwen_rows, qwen_done = load_completed_keys(QWEN_OUTPUT_CSV)

gen_counter = 0
run_start   = time.time()

print(f"Qwen: {TOTAL_PAIRS} pairs x {TOTAL_VARIANTS} variants = {TOTAL_GENS} total")
print(f"Done so far: {len(qwen_done)} | Remaining: {TOTAL_GENS - len(qwen_done)}")

for pair in sampled_pairs:
    pair_id   = pair["pair_id"]
    ethnicity = pair["ethnicity"]
    img1_path = pair["image1_path"]
    img2_path = pair["image2_path"]
    gt_label  = pair["ground_truth_label"]
    margin_arc = pair["verifier_score_margin_arcface"]
    margin_ada = pair["verifier_score_margin_adaface"]

    for vnum, (vlabel, vtext) in PROMPT_VARIANTS.items():
        gen_counter += 1

        if (pair_id, vnum) in qwen_done:
            continue

        ts  = datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z"
        out, gen_time = qwen_generate(qwen_model, qwen_processor, img1_path, img2_path, vtext)

        v     = has_verdict_v2(out)
        trunc = output_is_truncated(out)
        coh   = check_coherence(out)

        qwen_rows.append({
            "pair_id"               : pair_id,
            "ethnicity"             : ethnicity,
            "image1_filename"       : pair["image1_filename"],
            "image2_filename"       : pair["image2_filename"],
            "ground_truth_label"    : gt_label,
            "verifier_score_margin_arcface" : margin_arc,
            "verifier_score_margin_adaface" : margin_ada,
            "model_name"            : "Qwen2.5-VL-7B-Instruct",
            "prompt_variant"        : vnum,
            "prompt_variant_label"  : vlabel,
            "model_output"          : out,
            "inference_time_seconds": round(gen_time, 3),
            "word_count"            : len(out.split()),
            "coherence_passed"      : coh,
            "verdict_detected"      : v["verdict_detected"],
            "verdict_tier"          : v["verdict_tier"],
            "verdict_polarity"      : v["verdict_polarity"],
            "is_truncated"          : trunc,
            "status"                : compute_status(v["verdict_detected"], v["verdict_tier"], trunc),
            "timestamp"             : ts,
        })
        qwen_done.add((pair_id, vnum))

        # Save every SAVE_EVERY=10 new rows. /kaggle/working/ survives in-session
        # kernel restarts; on a full new session the checkpoint is gone, so frequent
        # saves keep potential re-work bounded.
        if len(qwen_rows) % SAVE_EVERY == 0:
            pd.DataFrame(qwen_rows).to_csv(QWEN_OUTPUT_CSV, index=False)

        if gen_counter % PROGRESS_EVERY == 0:
            elapsed_total = time.time() - run_start
            done_count    = len(qwen_done)
            eta_sec       = (elapsed_total / done_count) * (TOTAL_GENS - done_count) if done_count else 0
            print(
                f"[{done_count:3d}/{TOTAL_GENS}]  "
                f"{elapsed_total/60:.1f}min elapsed  "
                f"ETA {eta_sec/60:.1f}min  "
                f"pair={pair_id} var={vnum} {v['verdict_tier']} {gen_time:.1f}s"
            )

pd.DataFrame(qwen_rows).to_csv(QWEN_OUTPUT_CSV, index=False)
print(f"\nQwen done. {len(qwen_rows)} rows -> {QWEN_OUTPUT_CSV}")

/tmp/ipykernel_58/495417913.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts  = datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z"
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen: 240 pairs x 2 variants = 480 total
Done so far: 0 | Remaining: 480
[ 20/480]  3.4min elapsed  ETA 78.5min  pair=10 var=2 literal 11.5s
[ 40/480]  6.7min elapsed  ETA 74.1min  pair=20 var=2 literal 16.6s
[ 60/480]  10.0min elapsed  ETA 70.2min  pair=30 var=2 literal 12.0s
[ 80/480]  13.4min elapsed  ETA 67.0min  pair=40 var=2 literal 13.5s
[100/480]  16.6min elapsed  ETA 63.1min  pair=50 var=2 literal 15.1s
[120/480]  20.2min elapsed  ETA 60.6min  pair=60 var=2 literal 19.0s
[140/480]  23.5min elapsed  ETA 57.0min  pair=70 var=2 literal 15.1s
[160/480]  26.8min elapsed  ETA 53.6min  pair=80 var=2 literal 21.0s
[180/480]  30.2min elapsed  ETA 50.3min  pair=90 var=2 literal 15.8s
[200/480]  33.4min elapsed  ETA 46.8min  pair=100 var=2 literal 18.8s
[220/480]  36.9min elapsed  ETA 43.7min  pair=110 var=2 literal 14.7s
[240/480]  40.2min elapsed  ETA 40.2min  pair=120 var=2 literal 14.9s
[260/480]  43.6min elapsed  ETA 36.9min  pair=130 var=2 literal 14.2s
[280/480]  46.8min elapsed  

In [24]:
df_qwen_check = pd.read_csv(QWEN_OUTPUT_CSV)
n_rows = len(df_qwen_check)
total_expected = 240 * len(PROMPT_VARIANTS)
assert n_rows == total_expected, (
    f"Completion check failed: expected {total_expected} rows, got {n_rows} "
    f"({total_expected - n_rows} missing)."
)
print(f"Completion assertion passed: {n_rows}/{total_expected} rows.")

Completion assertion passed: 480/480 rows.


In [25]:
del qwen_model, qwen_processor
torch.cuda.empty_cache()
gc.collect()
print("Qwen unloaded.")

Qwen unloaded.


## 8 — Qwen Summary

Fresh load from disk — no shared state with the loop above.

In [26]:
df_qwen_summary = pd.read_csv(QWEN_OUTPUT_CSV)

print(f"Rows: {len(df_qwen_summary)}")
print()
print("-- Status --")
print(df_qwen_summary["status"].value_counts().to_string())
print()
print("-- Verdict tier (completed rows) --")
comp = df_qwen_summary["status"].isin(["completed_literal", "completed_semantic"])
if comp.any():
    print(pd.crosstab(df_qwen_summary[comp]["prompt_variant"],
                      df_qwen_summary[comp]["verdict_tier"]).to_string())
print()
print("-- Mean inference time (s) by variant --")
print(df_qwen_summary.groupby("prompt_variant")["inference_time_seconds"].mean().round(2).to_string())

Rows: 480

-- Status --
status
completed_literal    480

-- Verdict tier (completed rows) --
verdict_tier    literal
prompt_variant         
1                   240
2                   240

-- Mean inference time (s) by variant --
prompt_variant
1     4.79
2    15.19


---
## 9 — LLAVA-NEXT-MISTRAL-7B: Load, Run, Unload

> **Independent session note**: This block (§9–§10) can be run as its own Kaggle session.
> Prerequisite: `pairs_locked.csv` must be available at `/kaggle/working/pairs_locked.csv`
> (or mounted). Do not rerun §5. The checkpoint (`llava_next_full_results.csv`) persists
> within a session but is lost on a full new session — with `SAVE_EVERY=10` at most 10
> rows need to be regenerated.

### Why `tile_mode='single'` is non-negotiable

LLaVA-NeXT's default `image_grid_pinpoints` selects 672×672 tiling for a 400×400 image
→ 4 tiles + 1 thumbnail = ~2 880 visual tokens per image. Two images exceed T4 VRAM.
The fix overrides `image_grid_pinpoints` to `[[336, 336]]` on **both**
`processor.image_processor` and `model.config` (recursively), and `model.generate()` runs
**inside** the context. Exiting the context before generate restores the original
pinpoints mid-forward-pass and triggers a `split_with_sizes` mismatch crash.

In [10]:
# LLAVA_MEAN_SEC is computed from the LLaVA-NeXT pilot results CSV.
# Expected path (Kaggle): /kaggle/input/rfw-pilot-outputs/llava_next_pilot_results_annotated.csv
# Verified pilot mean: ~39.7s overall (V1: ~39.0s, V2: ~40.3s).
# Column used: inference_time_seconds, all rows (no error column in llava pilot schema).

LLAVA_PILOT_CSV = Path("/kaggle/input/datasets/chidroopakanaparthy/rfw-pilot-outputs/llava_next_pilot_results_annotated.csv")
LLAVA_FALLBACK_SEC = 39.7   # from verified pilot measurement; previous 35.0 was ~13% low

if LLAVA_PILOT_CSV.exists():
    _df_lp = pd.read_csv(LLAVA_PILOT_CSV)
    LLAVA_MEAN_SEC = round(_df_lp["inference_time_seconds"].dropna().mean(), 2)
    print(f"LLAVA_MEAN_SEC = {LLAVA_MEAN_SEC}s  (computed from {LLAVA_PILOT_CSV.name}, "
          f"n={len(_df_lp)} rows)")
else:
    LLAVA_MEAN_SEC = LLAVA_FALLBACK_SEC
    print(f"WARNING: Pilot CSV not found at {LLAVA_PILOT_CSV}")
    print(f"  Using verified pilot mean LLAVA_MEAN_SEC = {LLAVA_MEAN_SEC}s — mount the pilot "
          f"output dataset or update LLAVA_PILOT_CSV to compute from the file directly.")

pairs_for_estimate  = load_locked_pairs('/kaggle/input/datasets/chidroopakanaparthy/pairs-locked/pairs_locked.csv')
total_gens_estimate = len(pairs_for_estimate) * len(PROMPT_VARIANTS)
est_done            = len(load_completed_keys(LLAVA_OUTPUT_CSV)[1])
est_hours           = (total_gens_estimate * LLAVA_MEAN_SEC) / 3600
est_remaining_h     = ((total_gens_estimate - est_done) * LLAVA_MEAN_SEC) / 3600

print(f"\nLLaVA-NeXT: {total_gens_estimate} total generations @ {LLAVA_MEAN_SEC:.1f}s each")
print(f"  Full run estimate : {est_hours:.1f} h")
print(f"  Already done      : {est_done}")
print(f"  Remaining estimate: {est_remaining_h:.1f} h")
print(f"  Checkpoint cadence: every {SAVE_EVERY} rows")

LLAVA_MEAN_SEC = 39.68s  (computed from llava_next_pilot_results_annotated.csv, n=80 rows)

LLaVA-NeXT: 480 total generations @ 39.7s each
  Full run estimate : 5.3 h
  Already done      : 0
  Remaining estimate: 5.3 h
  Checkpoint cadence: every 10 rows


In [11]:
from transformers import LlavaNextForConditionalGeneration, LlavaNextProcessor

LLAVA_MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"

print(f"Loading {LLAVA_MODEL_ID} ...")
t0 = time.time()

llava_processor = LlavaNextProcessor.from_pretrained(LLAVA_MODEL_ID)

llava_model = LlavaNextForConditionalGeneration.from_pretrained(
    LLAVA_MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True,
)
llava_model.eval()

print(f"LLaVA-NeXT loaded in {time.time() - t0:.1f}s")
print(f"Processor pinpoints   : {llava_processor.image_processor.image_grid_pinpoints}")
print(f"Model.config pinpoints: {llava_model.config.image_grid_pinpoints}")

Loading llava-hf/llava-v1.6-mistral-7b-hf ...


processor_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

LLaVA-NeXT loaded in 105.0s
Processor pinpoints   : [[336, 672], [672, 336], [672, 672], [1008, 336], [336, 1008]]
Model.config pinpoints: [[336, 672], [672, 336], [672, 672], [1008, 336], [336, 1008]]


/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['model.image_newline']
  warnings.warn(


In [13]:
_SINGLE_TILE          = [[336, 336]]
_ORIG_PROC_PINPOINTS  = list(llava_processor.image_processor.image_grid_pinpoints)
_ORIG_MODEL_PINPOINTS = list(llava_model.config.image_grid_pinpoints)


def _patch_pinpoints(obj, val, visited=None):
    # Recursively set image_grid_pinpoints on nested config objects.
    # Transformers >=4.49 copies pinpoints into sub-configs (e.g. vision_tower.config)
    # that model.forward() reads independently.
    if visited is None:
        visited = set()
    if id(obj) in visited:
        return
    visited.add(id(obj))
    if hasattr(obj, "image_grid_pinpoints"):
        obj.image_grid_pinpoints = val
    for attr in ["config", "model", "vision_tower", "vision_model"]:
        child = getattr(obj, attr, None)
        if child is not None:
            _patch_pinpoints(child, val, visited)


@contextmanager
def single_tile_mode(proc, mdl):
    orig_proc = list(proc.image_processor.image_grid_pinpoints)
    try:
        proc.image_processor.image_grid_pinpoints = _SINGLE_TILE
        _patch_pinpoints(mdl, _SINGLE_TILE)
        yield
    finally:
        proc.image_processor.image_grid_pinpoints = orig_proc
        _patch_pinpoints(mdl, _ORIG_MODEL_PINPOINTS)


def llava_generate(model, processor, pil1, pil2, prompt_text):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Image 1.\n"},
                {"type": "image"},
                {"type": "text", "text": f"Image 2.\n{prompt_text}"},
            ],
        }
    ]

    # generate() is inside the context so the [[336,336]] override is active
    # during model.forward() -> get_image_features(), not just during tokenisation.
    with single_tile_mode(processor, model):
        prompt_str = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(
            images=[pil1, pil2],
            text=prompt_str,
            return_tensors="pt",
        ).to(model.device, dtype=DTYPE)

        t_start = time.perf_counter()
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )
        elapsed = time.perf_counter() - t_start

    input_len = inputs["input_ids"].shape[1]
    out = processor.decode(
        generated_ids[0, input_len:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()

    return out, elapsed


print(f"LLaVA helpers defined. Single-tile override: {_SINGLE_TILE}")

LLaVA helpers defined. Single-tile override: [[336, 336]]


In [16]:
# Reads pairs_locked.csv — does not touch random.Random or resample.
sampled_pairs = load_locked_pairs('')
TOTAL_PAIRS    = len(sampled_pairs)
TOTAL_VARIANTS = len(PROMPT_VARIANTS)
TOTAL_GENS     = TOTAL_PAIRS * TOTAL_VARIANTS

llava_rows, llava_done = load_completed_keys(LLAVA_OUTPUT_CSV)

gen_counter = 0
run_start   = time.time()

print(f"LLaVA-NeXT: {TOTAL_PAIRS} pairs x {TOTAL_VARIANTS} variants = {TOTAL_GENS} total")
print(f"Done so far: {len(llava_done)} | Remaining: {TOTAL_GENS - len(llava_done)}")

for pair in sampled_pairs:
    pair_id   = pair["pair_id"]
    ethnicity = pair["ethnicity"]
    img1_path = Path(pair["image1_path"])
    img2_path = Path(pair["image2_path"])
    gt_label  = pair["ground_truth_label"]
    margin_arc = pair["verifier_score_margin_arcface"]
    margin_ada = pair["verifier_score_margin_adaface"]

    pil1 = Image.open(img1_path).convert("RGB")
    pil2 = Image.open(img2_path).convert("RGB")

    for vnum, (vlabel, vtext) in PROMPT_VARIANTS.items():
        gen_counter += 1

        if (pair_id, vnum) in llava_done:
            continue

        ts  = datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z"
        out, gen_time = llava_generate(llava_model, llava_processor, pil1, pil2, vtext)

        v     = has_verdict_v2(out)
        trunc = output_is_truncated(out)
        coh   = check_coherence(out)

        llava_rows.append({
            "pair_id"               : pair_id,
            "ethnicity"             : ethnicity,
            "image1_filename"       : pair["image1_filename"],
            "image2_filename"       : pair["image2_filename"],
            "ground_truth_label"    : gt_label,
            "verifier_score_margin_arcface" : margin_arc,
            "verifier_score_margin_adaface" : margin_ada,
            "model_name"            : "LLaVA-NeXT-Mistral-7B",
            "prompt_variant"        : vnum,
            "prompt_variant_label"  : vlabel,
            "model_output"          : out,
            "inference_time_seconds": round(gen_time, 3),
            "word_count"            : len(out.split()),
            "coherence_passed"      : coh,
            "verdict_detected"      : v["verdict_detected"],
            "verdict_tier"          : v["verdict_tier"],
            "verdict_polarity"      : v["verdict_polarity"],
            "is_truncated"          : trunc,
            "status"                : compute_status(v["verdict_detected"], v["verdict_tier"], trunc),
            "timestamp"             : ts,
        })
        llava_done.add((pair_id, vnum))

        if len(llava_rows) % SAVE_EVERY == 0:
            pd.DataFrame(llava_rows).to_csv(LLAVA_OUTPUT_CSV, index=False)

        if gen_counter % PROGRESS_EVERY == 0:
            elapsed_total = time.time() - run_start
            done_count    = len(llava_done)
            eta_sec       = (elapsed_total / done_count) * (TOTAL_GENS - done_count) if done_count else 0
            print(
                f"[{done_count:3d}/{TOTAL_GENS}]  "
                f"{elapsed_total/60:.1f}min elapsed  "
                f"ETA {eta_sec/60:.1f}min  "
                f"pair={pair_id} var={vnum} {v['verdict_tier']} {gen_time:.1f}s"
            )

        torch.cuda.empty_cache()
        gc.collect()

pd.DataFrame(llava_rows).to_csv(LLAVA_OUTPUT_CSV, index=False)
print(f"\nLLaVA-NeXT done. {len(llava_rows)} rows -> {LLAVA_OUTPUT_CSV}")

LLaVA-NeXT: 240 pairs x 2 variants = 480 total
Done so far: 0 | Remaining: 480


/tmp/ipykernel_58/3474205923.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts  = datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z"
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `p

[ 20/480]  13.1min elapsed  ETA 300.6min  pair=10 var=2 literal 30.3s


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

[ 40/480]  25.4min elapsed  ETA 279.9min  pair=20 var=2 semantic 35.0s


...

[440/480]  297.9min elapsed  ETA 27.1min  pair=220 var=2 semantic 37.6s


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

[460/480]  310.7min elapsed  ETA 13.5min  pair=230 var=2 semantic 44.0s


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

[480/480]  322.8min elapsed  ETA 0.0min  pair=240 var=2 semantic 33.9s

LLaVA-NeXT done. 480 rows -> /kaggle/working/llava_next_full_results.csv


In [18]:
df_llava_check = pd.read_csv(LLAVA_OUTPUT_CSV)
n_rows = len(df_llava_check)
total_expected = 240 * len(PROMPT_VARIANTS)
assert n_rows == total_expected, (
    f"Completion check failed: expected {total_expected} rows, got {n_rows} "
    f"({total_expected - n_rows} missing)."
)
print(f"Completion assertion passed: {n_rows}/{total_expected} rows.")

Completion assertion passed: 480/480 rows.


In [19]:
del llava_model, llava_processor
torch.cuda.empty_cache()
gc.collect()
print("LLaVA-NeXT unloaded.")

LLaVA-NeXT unloaded.


## 10 — LLaVA-NeXT Summary

Fresh load from disk — no shared state with the loop above.

In [20]:
df_llava_summary = pd.read_csv(LLAVA_OUTPUT_CSV)

print(f"Rows: {len(df_llava_summary)}")
print()
print("-- Status --")
print(df_llava_summary["status"].value_counts().to_string())
print()
print("-- Verdict tier (completed rows) --")
comp = df_llava_summary["status"].isin(["completed_literal", "completed_semantic"])
if comp.any():
    print(pd.crosstab(df_llava_summary[comp]["prompt_variant"],
                      df_llava_summary[comp]["verdict_tier"]).to_string())
print()
print("-- Mean inference time (s) by variant --")
print(df_llava_summary.groupby("prompt_variant")["inference_time_seconds"].mean().round(2).to_string())

Rows: 480

-- Status --
status
completed_literal       282
completed_semantic      197
no_verdict_ambiguous      1

-- Verdict tier (completed rows) --
verdict_tier    literal  semantic
prompt_variant                   
1                   196        44
2                    86       153

-- Mean inference time (s) by variant --
prompt_variant
1    37.80
2    41.61


## 11 — Combined Schema Verification

Confirms both CSVs match the expected schema and can be concatenated.

In [22]:
EXPECTED_COLUMNS = [
    "pair_id", "ethnicity", "image1_filename", "image2_filename",
    "ground_truth_label",
    "verifier_score_margin_arcface", "verifier_score_margin_adaface",
    "model_name", "prompt_variant", "prompt_variant_label", "model_output",
    "inference_time_seconds", "word_count", "coherence_passed",
    "verdict_detected", "verdict_tier", "verdict_polarity",
    "is_truncated", "status", "timestamp",
]

df_q = pd.read_csv('/kaggle/input/datasets/chidroopakanaparthy/owen-full/qwen_full_results.csv')
df_l = pd.read_csv(LLAVA_OUTPUT_CSV)

for name, df in [("Qwen", df_q), ("LLaVA-NeXT", df_l)]:
    missing = set(EXPECTED_COLUMNS) - set(df.columns)
    extra   = set(df.columns) - set(EXPECTED_COLUMNS)
    print(f"{name}: {len(df)} rows | missing={missing} | extra={extra}")

print()
df_combined = pd.concat([df_q, df_l], ignore_index=True)
print(f"Combined: {len(df_combined)} rows, {df_combined['model_name'].nunique()} models")
print(df_combined.groupby(["model_name", "status"]).size().unstack(fill_value=0).to_string())

Qwen: 480 rows | missing=set() | extra=set()
LLaVA-NeXT: 480 rows | missing=set() | extra=set()

Combined: 960 rows, 2 models
status                  completed_literal  completed_semantic  no_verdict_ambiguous
model_name                                                                         
LLaVA-NeXT-Mistral-7B                 282                 197                     1
Qwen2.5-VL-7B-Instruct                480                   0                     0


In [2]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

try:
    qwen_df = pd.read_csv('/kaggle/input/datasets/chidroopakanaparthy/owen-full/qwen_full_results.csv')
    llava_df = pd.read_csv('/kaggle/input/datasets/chidroopakanaparthy/llava/llava_next_full_results.csv')
    df = pd.concat([qwen_df, llava_df], ignore_index=True)
except FileNotFoundError as e:
    print(f"Error loading CSVs: {e}")
    print("Ensure both Qwen and LLaVA generation cells have completed and saved their outputs.")
    raise

valid_polarities = ['same', 'different']
df = df[df['verdict_polarity'].isin(valid_polarities)].copy()

polarity_map = {'same': 'genuine', 'different': 'impostor'}
df['predicted_label'] = df['verdict_polarity'].map(polarity_map)

df['is_correct'] = (df['predicted_label'] == df['ground_truth_label']).astype(int)

summary_metrics = []
chi2_results = []
races = ["African", "Asian", "Caucasian", "Indian"]

def analyze_slice(slice_df, model_label, prompt_variant_label):
    contingency = []
    race_accuracies = {}
    
    for race in races:
        r_df = slice_df[slice_df['ethnicity'] == race]
        if len(r_df) == 0:
            continue
            
        n_pairs = len(r_df)
        gen_df = r_df[r_df['ground_truth_label'] == 'genuine']
        imp_df = r_df[r_df['ground_truth_label'] == 'impostor']
        
        n_genuine = len(gen_df)
        n_impostor = len(imp_df)
        
        tpr = gen_df['is_correct'].mean() if n_genuine > 0 else 0.0
        tnr = imp_df['is_correct'].mean() if n_impostor > 0 else 0.0
        fpr = 1.0 - tnr
        acc = r_df['is_correct'].mean()
        
        summary_metrics.append({
            'Race': race,
            'Model': model_label,
            'Prompt_Variant': prompt_variant_label,
            'Accuracy': round(acc, 4),
            'TPR': round(tpr, 4),
            'FPR': round(fpr, 4),
            'TNR': round(tnr, 4),
            'N_pairs': n_pairs,
            'N_genuine': n_genuine,
            'N_impostor': n_impostor
        })
        
        n_correct = int(r_df['is_correct'].sum())
        n_incorrect = n_pairs - n_correct
        contingency.append([n_correct, n_incorrect])
        race_accuracies[race] = acc
   
    if len(contingency) == 4:
        chi2, p_value, dof, expected = chi2_contingency(np.array(contingency))
        best_race = max(race_accuracies, key=race_accuracies.get)
        worst_race = min(race_accuracies, key=race_accuracies.get)
        
        chi2_results.append({
            'Model': model_label,
            'Prompt_Variant': prompt_variant_label,
            'Chi2': round(chi2, 4),
            'p_value': round(p_value, 6),
            'DoF': dof,
            'Significant': 'Yes' if p_value < 0.05 else 'No',
            'Best_Race': best_race,
            'Best_Acc': round(race_accuracies[best_race], 4),
            'Worst_Race': worst_race,
            'Worst_Acc': round(race_accuracies[worst_race], 4),
        })

for model_name in sorted(df['model_name'].unique()):
    m_df = df[df['model_name'] == model_name]
    
    analyze_slice(m_df, model_name, 'All')
    
    for variant in sorted(m_df['prompt_variant'].unique()):
        v_df = m_df[m_df['prompt_variant'] == variant]
        # Include the variant label name if available (e.g. "Direct" or "CoT")
        v_label = v_df['prompt_variant_label'].iloc[0] if 'prompt_variant_label' in v_df.columns else f'Variant {variant}'
        analyze_slice(v_df, model_name, f"V{variant}: {v_label}")

# 3. Save and Display Results

summary_df = pd.DataFrame(summary_metrics)  
chi2_df = pd.DataFrame(chi2_results)

summary_df.to_csv('/kaggle/working/vlm_summary_metrics.csv', index=False)
chi2_df.to_csv('/kaggle/working/vlm_chi2_results.csv', index=False)

print("=" * 80)
print("VLM CHI-SQUARE DISPARITY RESULTS (Matches baseline methodology)")
print("=" * 80)
display(chi2_df)

print("\n" + "=" * 80)
print("VLM SUMMARY METRICS PER ETHNICITY")
print("=" * 80)
display(summary_df)

print("\nResults saved to /kaggle/working/vlm_summary_metrics.csv and /kaggle/working/vlm_chi2_results.csv")

VLM CHI-SQUARE DISPARITY RESULTS (Matches baseline methodology)


,Model,Prompt_Variant,Chi2,p_value,DoF,Significant,Best_Race,Best_Acc,Worst_Race,Worst_Acc
0,LLaVA-NeXT-Mistral-7B,All,6.1663,0.103796,3,No,Indian,0.6167,Caucasian,0.4667
1,LLaVA-NeXT-Mistral-7B,V1: direct_feature_comparison,1.9382,0.585337,3,No,Indian,0.6000,Caucasian,0.4833
2,LLaVA-NeXT-Mistral-7B,V2: chain_of_thought,4.6179,0.202010,3,No,Indian,0.6333,Caucasian,0.4500
3,Qwen2.5-VL-7B-Instruct,All,7.7412,0.051674,3,No,Caucasian,0.7083,African,0.5500
4,Qwen2.5-VL-7B-Instruct,V1: direct_feature_comparison,3.1187,0.373685,3,No,Caucasian,0.6667,African,0.5333
5,Qwen2.5-VL-7B-Instruct,V2: chain_of_thought,7.7778,0.050835,3,No,Asian,0.7833,African,0.5667



VLM SUMMARY METRICS PER ETHNICITY


,Race,Model,Prompt_Variant,Accuracy,TPR,FPR,TNR,N_pairs,N_genuine,N_impostor
0,African,LLaVA-NeXT-Mistral-7B,All,0.5000,0.7500,0.7500,0.2500,120,60,60
1,Asian,LLaVA-NeXT-Mistral-7B,All,0.5042,0.8475,0.8333,0.1667,119,59,60
2,Caucasian,LLaVA-NeXT-Mistral-7B,All,0.4667,0.8333,0.9000,0.1000,120,60,60
3,Indian,LLaVA-NeXT-Mistral-7B,All,0.6167,0.8667,0.6333,0.3667,120,60,60
4,African,LLaVA-NeXT-Mistral-7B,V1: direct_feature_comparison,0.5167,0.8333,0.8000,0.2000,60,30,30
5,Asian,LLaVA-NeXT-Mistral-7B,V1: direct_feature_comparison,0.5000,0.8333,0.8333,0.1667,60,30,30
6,Caucasian,LLaVA-NeXT-Mistral-7B,V1: direct_feature_comparison,0.4833,0.8333,0.8667,0.1333,60,30,30
7,Indian,LLaVA-NeXT-Mistral-7B,V1: direct_feature_comparison,0.6000,0.9000,0.7000,0.3000,60,30,30
8,African,LLaVA-NeXT-Mistral-7B,V2: chain_of_thought,0.4833,0.6667,0.7000,0.3000,60,30,30
9,Asian,LLaVA-NeXT-Mistral-7B,V2: chain_of_thought,0.5085,0.8621,0.8333,0.1667,59,29,30



Results saved to /kaggle/working/vlm_summary_metrics.csv and /kaggle/working/vlm_chi2_results.csv


In [ ]:
import pandas as pd
from statsmodels.stats.multitest import multipletests

# 1. Read from your read-only input dataset
input_path = '/kaggle/input/datasets/chidroopakanaparthy/vlm-chi2/vlm_chi2_results.csv'
chi2_df = pd.read_csv(input_path)

# 2. Extract the raw p-values
p_values = chi2_df['p_value'].values

# 3. Apply Benjamini-Hochberg FDR correction across all 6 tests in one pass
reject, pvals_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

# 4. Add the new columns
chi2_df['p_value_fdr'] = [round(p, 6) for p in pvals_corrected]
chi2_df['Significant_FDR'] = ['Yes' if r else 'No' for r in reject]

# 5. Reorder columns
cols = chi2_df.columns.tolist()
sig_idx = cols.index('Significant')
new_cols = (
    cols[:sig_idx + 1] + 
    ['p_value_fdr', 'Significant_FDR'] + 
    [c for c in cols[sig_idx + 1:] if c not in ['p_value_fdr', 'Significant_FDR']]
)
chi2_df = chi2_df[new_cols]

# 6. Save the new file to the writeable /kaggle/working directory!
output_path = '/kaggle/working/vlm_chi2_results_fdr.csv'
chi2_df.to_csv(output_path, index=False)

print("=" * 100)
print("VLM CHI-SQUARE DISPARITY RESULTS (with Benjamini-Hochberg FDR Correction)")
print("=" * 100)
display(chi2_df)
print(f"\n Updated results saved to {output_path}")

In [4]:
"""
RUN THIS ON KAGGLE (where you have access to /kaggle/input/.../rfw-clean-400x400/).
Zips only the 478 images actually used in the annotation sample, so you're not
uploading the whole RFW dataset to Drive.

Steps:
1. Upload images_needed.csv to your Kaggle notebook (or regenerate it there from pairs_locked.csv).
2. Run this cell.
3. Download the output zip from Kaggle's output panel.
4. Unzip locally, then drag the folder into a Google Drive folder.
5. Right-click that Drive folder -> Share -> "Anyone with the link" -> Viewer.
"""
import pandas as pd
import shutil, os, glob

# Find images_needed.csv wherever Kaggle put it (working dir or an input dataset folder)
candidates = glob.glob('/kaggle/input/datasets/chidroopakanaparthy/images-needed/images_needed.csv', recursive=True) + \
             glob.glob('/kaggle/working/images_needed.csv')
if not candidates:
    raise FileNotFoundError(
        "images_needed.csv not found under /kaggle/input or /kaggle/working. "
        "Upload the file (columns: filename, path) as a Kaggle Dataset or notebook input."
    )
csv_path = candidates[0]
print(f"Loading: {csv_path}")

imgs = pd.read_csv(csv_path)
print("Columns found:", imgs.columns.tolist())

if 'path' not in imgs.columns:
    raise KeyError(
        f"Expected a 'path' column, got {imgs.columns.tolist()}. "
        "You likely uploaded pairs_locked.csv (which has image1_path/image2_path) "
        "instead of images_needed.csv (which has filename, path). Re-upload the correct file."
    )

os.makedirs('/kaggle/working/annotation_images', exist_ok=True)

missing = []
for _, row in imgs.iterrows():
    src = row['path']
    dst = f"/kaggle/working/annotation_images/{row['filename']}"
    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        missing.append(src)

print(f"Copied {len(imgs) - len(missing)} / {len(imgs)} images")
if missing:
    print("MISSING (check path):", missing[:5])

shutil.make_archive('/kaggle/working/annotation_images', 'zip', '/kaggle/working/annotation_images')
print("Zip ready: /kaggle/working/annotation_images.zip -- download it from the Output panel.")

Loading: /kaggle/input/datasets/chidroopakanaparthy/images-needed/images_needed.csv
Columns found: ['filename', 'path']
Copied 478 / 478 images
Zip ready: /kaggle/working/annotation_images.zip -- download it from the Output panel.
